# Chapter 5 — Mineral Dataset (Maronna et al. 2019)

Reproduces **Figures 5.1 – 5.7** of *Robust Statistics: Theory and Methods*, Maronna, Martin, Yohai & Salibián-Barrera (Wiley, 2019), using `robstattm_py` — the Python port of the RobStatTM R package.

The plots in this notebook are produced by **R** through `rpy2` (Path A in `docs/plotting_strategy.md`), so they are pixel-equivalent to the published figures.

The fitted model is `lmrobdetMM(zinc ~ copper)` with `bb=0.5`, `efficiency=0.85`, `family="bisquare"`. Comparison fits:

* **LS** — ordinary least squares on the full data
* **L1** — least-absolute-deviations (from `quantreg::rq`)
* **LS-15** — ordinary least squares with observation 15 removed
* **MM (Rob)** — `lmrobdetMM` robust fit on the full data

Every numeric output below is bit-for-bit identical to the R reference (`atol=0, rtol=0`).

## Setup

We need R available to `rpy2`. On Windows set `R_HOME` and prepend R's `bin\x64` to `PATH` before importing `robstattm_py`. The `quantreg` package supplies the L1 fit; install with `install.packages("quantreg")` if missing.

In [ ]:
import os, sys, pathlib

# Windows R_HOME setup (skip if already configured)
if sys.platform == "win32" and "R_HOME" not in os.environ:
    os.environ["R_HOME"] = r"C:\Program Files\R\R-4.5.2"
    os.environ["PATH"] = r"C:\Program Files\R\R-4.5.2\bin\x64;" + os.environ["PATH"]

import robstattm_py as rpm
from robstattm_py.plotting import r_plot, show_png
from robstattm_py._r import r as _r

FIG_DIR = pathlib.Path("figures")
FIG_DIR.mkdir(exist_ok=True)

print(f"robstattm_py {rpm.__version__}")

## The mineral dataset

53 paired measurements of zinc and copper concentrations in mineral samples. Observation 15 is a known leverage point.

In [ ]:
mineral = rpm.datasets.mineral()
print(mineral.shape)
mineral.head()

## Fit four models in R

We pre-fit LS, L1, LS-15, and MM on the R side so the subsequent plot expressions can reference them by name. (For end-to-end Python, see the strict-tier wrapper tests in `tests/regression/`.)

In [ ]:
ro = _r()
ro.r("""
library(RobStatTM)
data(mineral)

cont <- lmrobdet.control(bb = 0.5, efficiency = 0.85, family = "bisquare")

mineralls    <- lm(zinc ~ copper, data = mineral)
minerall1    <- quantreg::rq(zinc ~ copper, data = mineral)
minerallssin <- lm(zinc ~ copper, data = mineral[-15, ])
mineralMM    <- lmrobdetMM(zinc ~ copper, data = mineral, control = cont)
""")
print("R fits ready: mineralls, minerall1, minerallssin, mineralMM")

## Cross-check: Python wrapper matches R

Before plotting, verify that the Python `lmrobdet_mm` call produces byte-identical coefficients to the R fit above. This is the strict-tier promise made by `robstattm_py`.

In [ ]:
import numpy as np

ctrl = rpm.lmrobdet_control(bb=0.5, efficiency=0.85, family="bisquare")
py_fit = rpm.lmrobdet_mm("zinc ~ copper", data=mineral, control=ctrl)

r_coef = np.asarray(ro.r("coef(mineralMM)"), dtype=float)
print("R   coefficients :", r_coef)
print("Py  coefficients :", py_fit.coefficients)
print("bit-equal        :", np.array_equal(py_fit.coefficients, r_coef))

py_fit

## Figure 5.1 — LS / L1 / LS-15 on the full data

The three classical lines all bend toward observation 15 in different amounts. Note that LS is the most distorted.

In [ ]:
show_png(r_plot("""
plot(zinc ~ copper, data=mineral, pch=19, cex=1.3)
abline(mineralls,    lwd=2, col='red')
abline(minerall1,    lwd=2, col='blue')
abline(minerallssin, lwd=2, col='green4')
text(c(600,600,600), c(29,55,82), c('LS-15','L1','LS'), cex=1.3)
text(mineral[15,1], mineral[15,2]-6, '15', cex=1.3)
""", path=FIG_DIR / "fig_5_1.png"))

## Figure 5.2 — QQ-plot of LS residuals

Observation 15 stands out heavily; observation 53 trails off.

In [ ]:
show_png(r_plot("""
plot(mineralls, which=2, add.smooth=FALSE, pch=19, id.n=2, cex.id=1.2)
abline(h=c(2.5, 0, -2.5), lty=2, lwd=2)
""", path=FIG_DIR / "fig_5_2.png"))

## Figure 5.3 — Residuals vs. fitted (LS)

Bands at ±2.5σ_LS show the diagnostic threshold.

In [ ]:
show_png(r_plot("""
sigmaLS <- summary(mineralls)$sigma
plot(mineralls, which=1, add.smooth=FALSE, pch=19, id.n=2, cex.id=1.2)
abline(h=c(2.5, 0, -2.5) * sigmaLS, lty=2, lwd=2)
""", path=FIG_DIR / "fig_5_3.png"))

## Figure 5.4 — MM-fit vs LS-15

The robust MM fit (`magenta`) tracks the LS-without-outlier line (`green`) closely on the full data — automatic outlier handling.

In [ ]:
show_png(r_plot("""
plot(zinc ~ copper, data=mineral, pch=19, cex=1.3)
abline(minerallssin, lwd=2, col='green4')
abline(mineralMM,    lwd=2, col='magenta')
text(c(600,600), c(45,29), c('ROB','LS-15'), cex=1.2)
text(mineral[15,1], mineral[15,2]-6, '15', cex=1.2)
""", path=FIG_DIR / "fig_5_4.png"))

## Figure 5.5 — MM diagnostic plot (which=4)

`plot.lmrobdetMM`'s `which=4` panel: case-wise robust weights — observations with weight near zero are downweighted outliers.

In [ ]:
show_png(r_plot("""
plot(mineralMM, which=4, add.smooth=FALSE, pch=19, cex.id=1.3)
""", path=FIG_DIR / "fig_5_5.png"))

## Figure 5.6 — MM residuals QQ-plot

Bands at ±2.5 × robust scale identify outliers automatically.

In [ ]:
show_png(r_plot("""
plot(mineralMM, which=2, pch=19, id.n=3, cex.id=1.2)
abline(h=c(-2.5, 0, 2.5) * mineralMM$scale, lty=2, lwd=2)
""", path=FIG_DIR / "fig_5_6.png"))

## Figure 5.7 — LS vs robust residuals

Sorted absolute residuals (excluding the largest one) for the LS vs MM fits. The y=x line (red) shows where they would agree; the MM residuals are uniformly smaller in the bulk.

In [ ]:
show_png(r_plot("""
plot(sort(abs(resid(mineralls)))[-53],
     sort(abs(resid(mineralMM)))[-53],
     xlab='Least Squares residuals', ylab='Robust residuals',
     pch=19, cex=1.3)
abline(0, 1, lwd=2, col='red')
""", path=FIG_DIR / "fig_5_7.png"))

## Table 5.1 — Pure-Python summary of the MM fit

Same numerical values that `summary(mineralMM)` produces in R, rendered through the Python wrapper's `.summary()` method.

In [ ]:
summ = py_fit.summary()
summ.coefficients_table

In [ ]:
print(f"Robust residual standard error: {summ.scale:.4f}")
print(f"Multiple R-squared            : {summ.r_squared:.5f}")
print(f"Adjusted R-squared            : {summ.adj_r_squared:.5f}")
print(f"IRWLS iterations              : {summ.iter}")
print(f"Converged                     : {summ.converged}")
print(f"Hat-values (max, min)         : "
      f"{py_fit.hatvalues().max():.4f}, {py_fit.hatvalues().min():.4f}")
print(f"RFPE (scalar / both)          : "
      f"{py_fit.rfpe():.4f} / {py_fit.rfpe(both_vals=True)}")